# Paso 2 · Target y partición común (80/20)

**Qué hace este notebook**
1. Define la población de trabajo: los préstamos con resultado conocido (*Fully Paid* y *Charged Off*).
2. Crea la variable objetivo `default`.
3. Genera **una sola vez** la partición 80/20 estratificada con semilla fija y la guarda en Drive.

**Por qué importa:** la prueba de DeLong compara AUC calculados sobre *las mismas observaciones*. Si scikit-learn y PySpark usaran particiones distintas, la comparación no sería válida. Por eso la partición se calcula una vez y los dos entornos la leen.

## 1. Cargar el dataset completo

In [ ]:
import os, glob, time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import drive

drive.mount('/content/drive')
RUTA = '/content/drive/MyDrive/ML_LendingClub'   # <- ajusta si tu carpeta se llama distinto

archivos = sorted(f for f in glob.glob(RUTA + '/*accepted*') if os.path.isfile(f))
print('Archivo a leer:', archivos)

t0 = time.time()
df = pd.read_csv(archivos[0], low_memory=False)
print(f'Carga: {time.time() - t0:,.1f} s | dimensión: {df.shape}')

Mounted at /content/drive
Archivo a leer: ['/content/drive/MyDrive/ML_LendingClub/accepted_2007_to_2018Q4.csv.gz']
Carga: 105.2 s | dimensión: (2260701, 151)


## 2. Población de trabajo y variable objetivo
En el notebook anterior vimos 2.260.701 filas, pero solo **1.345.310** tienen un resultado conocido (*Fully Paid* o *Charged Off*). Los préstamos vigentes (*Current*) o con mora aún no sabemos cómo terminan, y marcarlos como `default = 0` sería afirmar algo que no se sabe.

Filtrar por estado **no es muestrear**: es definir qué filas tienen una etiqueta válida. Se usa todo ese conjunto, sin submuestreo. Esta decisión debe documentarse en el informe.

Para el target usamos una comparación vectorizada en lugar de `apply(lambda ...)`: da el mismo resultado y es mucho más rápida con más de un millón de filas.

In [ ]:
ESTADOS_CERRADOS = ['Fully Paid', 'Charged Off']
df = df[df['loan_status'].isin(ESTADOS_CERRADOS)].copy()
df = df.reset_index(drop=True)
print('Filas tras filtrar préstamos cerrados:', len(df))

# Target: 1 = Charged Off (default), 0 = Fully Paid
df['default'] = (df['loan_status'] == 'Charged Off').astype(int)

Filas tras filtrar préstamos cerrados: 1345310


In [ ]:
dist = df['default'].value_counts().sort_index()
print(dist)
print()
print((df['default'].value_counts(normalize=True).sort_index() * 100).round(2).astype(str) + ' %')

default
0    1076751
1     268559
Name: count, dtype: int64

default
0    80.04 %
1    19.96 %
Name: proportion, dtype: object


## 3. Identificador único
Spark **no garantiza el orden de las filas** al leer un CSV, así que la partición no puede guardarse por posición: debe unirse por una columna que esté en los datos. Usamos `id`, que debe ser único y no nulo.

In [ ]:
print('id nulos:', int(df['id'].isna().sum()))
print('id únicos:', df['id'].is_unique)
assert df['id'].notna().all() and df['id'].is_unique, 'El id no es único: avísame antes de seguir'
df['id'] = df['id'].astype(str)

id nulos: 0
id únicos: True


## 4. Partición 80/20 estratificada
- `test_size=0.2`: 80 % entrenamiento, 20 % prueba.
- `stratify=df['default']`: conserva la misma proporción de defaults en ambos conjuntos (importante porque la clase positiva es minoritaria).
- `random_state=42`: la partición se puede reproducir exactamente.

In [ ]:
idx_train, idx_test = train_test_split(
    df.index, test_size=0.2, stratify=df['default'], random_state=42
)
split = pd.Series('train', index=df.index)
split.loc[idx_test] = 'test'

particion = pd.DataFrame({'id': df['id'].values, 'split': split.values})

In [ ]:
resumen = df.assign(split=split.values).groupby('split')['default'].agg(filas='count', tasa_default='mean')
resumen['tasa_default'] = (resumen['tasa_default'] * 100).round(3)
resumen

,filas,tasa_default
split,,
test,269062,19.963
train,1076248,19.963


Las dos tasas de default deben ser prácticamente iguales: esa es la prueba de que la estratificación funcionó.

## 5. Guardar la partición en Drive
El archivo tendrá solo dos columnas, `id` y `split`. Después lo releemos para comprobar que se guardó bien.

In [ ]:
ruta_split = RUTA + '/split_80_20_seed42.parquet'
particion.to_parquet(ruta_split, index=False)

chequeo = pd.read_parquet(ruta_split)
print('Filas guardadas:', len(chequeo))
print(chequeo['split'].value_counts())
print('Coincide con la partición original:', bool((chequeo['id'].values == particion['id'].values).all()
                                                     and (chequeo['split'].values == particion['split'].values).all()))

Filas guardadas: 1345310
split
train    1076248
test      269062
Name: count, dtype: int64
Coincide con la partición original: True
